In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_6 import BioJepa, BioJepaConfig
from dataloader_v0_6 import PretrainLoader, AlignmentLoader, TrainingLoader
from training_v0_6 import create_model, load_feature_banks, run_pretraining, run_alignment, run_full_training, train_linear_decoder, maybe_compile
from config_v0_6 import PretrainConfig, AlignmentConfig, FullTrainingConfig, DecoderConfig, DataConfig
from evals.evals import EvalContext, run_pretraining_evals, run_alignment_evals, run_full_model_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('/Users/djemec/data/jepa/v0_6')
ref_root = Path('/Users/djemec/data/jepa/reference_data')

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoints',
    ref_dir = ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cpu


In [3]:
# Model architecture
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=2,
    heads=2,
    embed_dim=8,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.6,
    gaussian_scale=2.0,
    film_linear_multiple=1.0,
    sim_coeff=25.0,
    std_coeff=25.0,
    cov_coeff=1.0,
    pert_latent_dim= 8,
    pert_mode_dim= 8,
)

# Training configs
pt_cfg = PretrainConfig(epochs=1, lr=1e-3, batch_size=128) 
align_cfg = AlignmentConfig(epochs=100, lr=4e-3, batch_size=32)
full_cfg = FullTrainingConfig(epochs=1, predictor_lr=1e-3, batch_size=32) 
decoder_cfg = DecoderConfig(epochs=1, lr=1e-3, batch_size=16) 

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters() if p.requires_grad):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters() if p.requires_grad):,}')

Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])
Student/Teacher: 84,074
ACpredictor: 84,864
PerturbationComposer: 26,152


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_full_final.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

# added for mapping
key_mappings = {
    'student.linear_scaler.weight': 'student.expr_scaler.weight',
    'teacher.linear_scaler.weight': 'teacher.expr_scaler.weight'
}
for old_key, new_key in key_mappings.items():
    if old_key in state_dict:
        state_dict[new_key] = state_dict.pop(old_key)

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': pt_cfg.batch_size, 'seed': SEED
})
pt_eval_results = run_pretraining_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'pretraining_eval_report.json')
pt_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [ ]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': align_cfg.batch_size, 'seed': SEED
})
align_eval_results = run_alignment_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'alignment_eval_report.json')
align_eval_results

In [ ]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [6]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_6_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
decoder.load_state_dict(decoder_sd)

<All keys matched successfully>

In [7]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': full_cfg.batch_size, 'seed': SEED, 
    'test_total_examples': 20000,
})
full_eval_results = run_full_model_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'testing_model_eval_report.json')
full_eval_results

Using cpu
Loaded cached test inference from /Users/djemec/data/jepa/v0_6/test_inference_cache (8 shards, delete directory to recompute)
expression_prediction: Pearson=0.9224, R2=0.8143, Centroid_acc=0.0088
gene_level_analysis: Dir_acc=0.8308, Top50_acc=0.2155
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


perturbation_retrieval (dna): 100%|█████████████████████████████████| 5/5 [17:00<00:00, 204.09s/it]


perturbation_retrieval (dna): MRR=0.0031


perturbation_retrieval (chemical): 100%|█████████████████████████████| 5/5 [00:20<00:00,  4.20s/it]


perturbation_retrieval (chemical): MRR=0.2079
uncertainty_calibration: ECE=0.1565, Monotonicity=66.67%
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
moa_matching expression: Within=0.6239, Between=0.5794, Gap=0.0445, Ratio=1.0768x
dose_response: monotonicity=50.31%, spearman=-0.1037
Saved report to /Users/djemec/data/jepa/v0_6/eval_results/testing_model_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1102,
   'genes': 10000,
   'test_samples': 20000},
  'sample_level': {'mse': 0.285750463642925,
   'pearson_r_top20': 0.6310555506885052},
  'perturbation_level': {'r2_all_genes': {'mean': 0.8143288587809907,
    'median': 0.8588858246803284},
   'r2_top50_degs': {'mean': -0.15887036800600873,
    'median': -0.014492213726043701},
   'mse': {'mean': 0.038343120366334915, 'median': 0.031047089025378227},
   'pearson_all_genes': {'mean': 0.9223874012268607,
    'median': 0.9431758224964142},
   'pearson_delta_all_genes': {'mean': 0.31431451031812274,
    'median': 0.33284829556941986},
   'pearson_top50_degs': {'mean': 0.5338874803281334,
    'median': 0.5768158733844757}},
  'centroid_accuracy': {'accuracy': 0.008823529411764706, 'n_groups': 1020},
  'vs_baseline': {'beat_rate': 0.2595281306715064, 'n_evaluated': 1102},
  'severity': {'pearson_r': 0.6929764747619629,
   'spearman_r': 0.4683577746221615},
  'error_by_magnitude'

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()